# SkillScope — Documentación del Pipeline

> **Proyecto:** Análisis de tendencias de habilidades en el mercado laboral tecnológico  
> **Dataset:** 1.3M LinkedIn Jobs & Skills (2024) — Kaggle  
> **Alcance:** Ofertas de trabajo en Estados Unidos

---

## Contenido

1. [Configuración e imports](#1-configuración-e-imports)
2. [ETL — Extract](#2-etl--extract)
3. [ETL — Transform](#3-etl--transform)
4. [ETL — Load](#4-etl--load)
5. [EDA — Análisis Exploratorio](#5-eda--análisis-exploratorio)
   - 5.1 Top 20 Skills generales
   - 5.2 Distribución por nivel y modalidad
   - 5.3 Top Skills técnicas normalizadas
   - 5.4 Top 15 empresas contratando
   - 5.5 Skills por nivel de experiencia
   - 5.6 Tendencia semanal de skills
   - 5.7 Top combinaciones de skills
   - 5.8 Skills por modalidad de trabajo
6. [Conclusiones](#6-conclusiones)

---
## 1. Configuración e imports

In [ ]:
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# Estilo de visualizaciones
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Rutas del proyecto
DATA_RAW   = Path('../data/raw')
DATA_DB    = Path('../data/jobs.duckdb')

print('✅ Configuración lista')
print(f'   DuckDB path: {DATA_DB}')

---
## 2. ETL — Extract

Cargamos los dos archivos principales del dataset de Kaggle:
- `linkedin_job_postings.csv`: metadatos de cada oferta (título, empresa, ubicación, fecha, nivel)
- `job_skills.csv`: relación de cada oferta con sus skills requeridas

In [ ]:
# Cargar archivos raw
df_jobs = pd.read_csv(DATA_RAW / 'linkedin_job_postings.csv')
df_skills = pd.read_csv(DATA_RAW / 'job_skills.csv')

print('📦 Datos cargados:')
print(f'   - job_postings : {len(df_jobs):,} registros | {df_jobs.shape[1]} columnas')
print(f'   - job_skills   : {len(df_skills):,} registros | {df_skills.shape[1]} columnas')
print()
print('Columnas job_postings:', list(df_jobs.columns))
print('Columnas job_skills  :', list(df_skills.columns))

In [ ]:
# Vista previa de los datos
display(df_jobs.head(3))
display(df_skills.head(3))

---
## 3. ETL — Transform

Transformaciones aplicadas:
1. **Filtrado por país** → solo Estados Unidos
2. **Conversión de fechas** → columna `listed_date` y `week`
3. **Limpieza de texto** → strip en títulos y empresas
4. **Explosión de skills** → una fila por par (job, skill)
5. **Verificación de duplicados**

In [ ]:
# 1. Filtrar a Estados Unidos
df_us = df_jobs[df_jobs['job_location'].str.contains('United States', na=False)].copy()
print(f'Total registros    : {len(df_jobs):,}')
print(f'Filtrado US        : {len(df_us):,}')
print(f'Registros excluidos: {len(df_jobs) - len(df_us):,}')

In [ ]:
# 2. Conversión de fechas
df_us['listed_date'] = pd.to_datetime(df_us['original_listed_time'], unit='ms')
df_us['week'] = df_us['listed_date'].dt.to_period('W').astype(str)

# 3. Limpieza de texto
df_us['title'] = df_us['title'].str.strip()
df_us['company'] = df_us['company'].str.strip()

print('✅ Fechas convertidas')
print(f'   Rango: {df_us["listed_date"].min().date()} → {df_us["listed_date"].max().date()}')
print(f'   Semanas únicas: {df_us["week"].nunique()}')

In [ ]:
# 4. Explotar skills — una fila por (job_link, skill)
df_skills_us = df_skills[df_skills['job_link'].isin(df_us['job_link'])].copy()

# Las skills pueden venir como lista separada por comas en algunos datasets
# Si ya están en formato long (una skill por fila), no es necesario explotar
df_job_skills = df_skills_us.copy()
df_job_skills['skill'] = df_job_skills['job_skills'].str.strip().str.lower()
df_job_skills = df_job_skills[['job_link', 'skill']].drop_duplicates()

print(f'✅ Relaciones job→skill: {len(df_job_skills):,}')

# 5. Verificar duplicados
dupes_jobs   = df_us.duplicated(subset='job_link').sum()
dupes_skills = df_job_skills.duplicated().sum()
print(f'   Duplicados en jobs  : {dupes_jobs}')
print(f'   Duplicados en skills: {dupes_skills}')

---
## 4. ETL — Load

Cargamos los datos transformados en **DuckDB**, una base de datos OLAP embebida ideal para consultas analíticas sobre grandes volúmenes de datos.

**Esquema:**
- `jobs(job_link PK, title, company, job_location, experience_level, work_type, listed_date, week)`
- `job_skills(job_link, skill, PRIMARY KEY (job_link, skill))`

In [ ]:
# Conectar / crear base de datos
con = duckdb.connect(str(DATA_DB))

# Tabla jobs
con.execute('DROP TABLE IF EXISTS job_skills')
con.execute('DROP TABLE IF EXISTS jobs')

cols_jobs = ['job_link', 'title', 'company', 'job_location',
             'formatted_experience_level', 'work_type', 'listed_date', 'week']

con.execute("""
    CREATE TABLE jobs AS
    SELECT * FROM df_us
    LIMIT 0
""")

con.register('df_us_view', df_us[cols_jobs])
con.execute('CREATE TABLE jobs AS SELECT * FROM df_us_view')

# Tabla job_skills
con.register('df_skills_view', df_job_skills)
con.execute('CREATE TABLE job_skills AS SELECT * FROM df_skills_view')

# Verificar carga
n_jobs   = con.execute('SELECT COUNT(*) FROM jobs').fetchone()[0]
n_skills = con.execute('SELECT COUNT(*) FROM job_skills').fetchone()[0]

print('✅ Carga a DuckDB completada:')
print(f'   Tabla jobs      : {n_jobs:,} registros')
print(f'   Tabla job_skills: {n_skills:,} registros')

---
## 5. EDA — Análisis Exploratorio

Con los datos ya en DuckDB, realizamos consultas SQL para obtener insights del mercado laboral.
Todas las visualizaciones usan datos directamente desde la base de datos.

### 5.1 Top 20 Skills generales

In [ ]:
top_skills = con.execute("""
    SELECT skill, COUNT(*) as frecuencia
    FROM job_skills
    GROUP BY skill
    ORDER BY frecuencia DESC
    LIMIT 20
""").df()

fig, ax = plt.subplots(figsize=(12, 7))
sns.barplot(data=top_skills, y='skill', x='frecuencia', palette='Blues_r', ax=ax)
ax.set_title('Top 20 Skills más demandadas (LinkedIn US 2024)', fontsize=14, fontweight='bold')
ax.set_xlabel('Número de ofertas')
ax.set_ylabel('Skill')
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
plt.tight_layout()
plt.show()

### 5.2 Distribución por nivel de experiencia y modalidad

In [ ]:
exp_dist = con.execute("""
    SELECT formatted_experience_level as nivel,
           COUNT(*) as total
    FROM jobs
    WHERE formatted_experience_level IS NOT NULL
    GROUP BY nivel
    ORDER BY total DESC
""").df()

work_dist = con.execute("""
    SELECT work_type as modalidad,
           COUNT(*) as total
    FROM jobs
    WHERE work_type IS NOT NULL
    GROUP BY modalidad
    ORDER BY total DESC
""").df()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].pie(exp_dist['total'], labels=exp_dist['nivel'],
            autopct='%1.1f%%', colors=sns.color_palette('muted'))
axes[0].set_title('Distribución por nivel de experiencia', fontweight='bold')

axes[1].pie(work_dist['total'], labels=work_dist['modalidad'],
            autopct='%1.1f%%', colors=sns.color_palette('pastel'))
axes[1].set_title('Distribución por modalidad de trabajo', fontweight='bold')

plt.tight_layout()
plt.show()

### 5.3 Top Skills técnicas normalizadas

Filtramos skills técnicas clave excluyendo soft skills genéricas.

In [ ]:
TECH_SKILLS = [
    'python', 'sql', 'java', 'javascript', 'excel', 'r',
    'data analysis', 'machine learning', 'aws', 'azure', 'gcp',
    'docker', 'kubernetes', 'spark', 'tableau', 'power bi',
    'tensorflow', 'pytorch', 'git', 'linux'
]

tech_query = con.execute(f"""
    SELECT skill, COUNT(*) as frecuencia
    FROM job_skills
    WHERE skill IN ({', '.join([repr(s) for s in TECH_SKILLS])})
    GROUP BY skill
    ORDER BY frecuencia DESC
""").df()

fig, ax = plt.subplots(figsize=(12, 7))
colors = sns.color_palette('viridis', len(tech_query))
sns.barplot(data=tech_query, y='skill', x='frecuencia', palette=colors, ax=ax)
ax.set_title('Top Skills Técnicas (normalizadas)', fontsize=14, fontweight='bold')
ax.set_xlabel('Número de ofertas')
ax.set_ylabel('Skill')
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
plt.tight_layout()
plt.show()

### 5.4 Top 15 empresas contratando

In [ ]:
top_companies = con.execute("""
    SELECT company, COUNT(*) as ofertas
    FROM jobs
    WHERE company IS NOT NULL
    GROUP BY company
    ORDER BY ofertas DESC
    LIMIT 15
""").df()

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=top_companies, y='company', x='ofertas', palette='rocket_r', ax=ax)
ax.set_title('Top 15 Empresas con más ofertas de trabajo (US)', fontsize=14, fontweight='bold')
ax.set_xlabel('Número de ofertas')
ax.set_ylabel('Empresa')
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
plt.tight_layout()
plt.show()

### 5.5 Skills por nivel de experiencia

Comparamos qué skills se requieren en niveles Mid-Senior vs Associate.

In [ ]:
skills_by_level = con.execute("""
    SELECT js.skill, j.formatted_experience_level as nivel, COUNT(*) as total
    FROM job_skills js
    JOIN jobs j ON js.job_link = j.job_link
    WHERE j.formatted_experience_level IN ('Mid-Senior level', 'Associate')
      AND js.skill IN (SELECT skill FROM job_skills GROUP BY skill ORDER BY COUNT(*) DESC LIMIT 15)
    GROUP BY js.skill, nivel
""").df()

pivot = skills_by_level.pivot(index='skill', columns='nivel', values='total').fillna(0)

fig, ax = plt.subplots(figsize=(12, 7))
pivot.plot(kind='barh', ax=ax, colormap='Set2')
ax.set_title('Top Skills por Nivel de Experiencia', fontsize=14, fontweight='bold')
ax.set_xlabel('Número de ofertas')
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
plt.tight_layout()
plt.show()

### 5.6 Tendencia semanal de skills

In [ ]:
TOP_N_TREND = ['python', 'sql', 'data analysis', 'excel', 'aws']

trend = con.execute(f"""
    SELECT j.week, js.skill, COUNT(*) as total
    FROM job_skills js
    JOIN jobs j ON js.job_link = j.job_link
    WHERE js.skill IN ({', '.join([repr(s) for s in TOP_N_TREND])})
    GROUP BY j.week, js.skill
    ORDER BY j.week
""").df()

pivot_trend = trend.pivot(index='week', columns='skill', values='total').fillna(0)

fig, ax = plt.subplots(figsize=(14, 6))
pivot_trend.plot(ax=ax, marker='o', linewidth=2)
ax.set_title('Tendencia semanal de Top Skills Técnicas', fontsize=14, fontweight='bold')
ax.set_xlabel('Semana')
ax.set_ylabel('Número de ofertas')
ax.tick_params(axis='x', rotation=45)
ax.legend(title='Skill')
plt.tight_layout()
plt.show()

### 5.7 Top combinaciones de skills

Skills que aparecen frecuentemente juntas en la misma oferta.

In [ ]:
combos = con.execute("""
    SELECT a.skill as skill_1, b.skill as skill_2, COUNT(*) as co_ocurrencias
    FROM job_skills a
    JOIN job_skills b ON a.job_link = b.job_link AND a.skill < b.skill
    WHERE a.skill IN ('python','sql','data analysis','excel','aws','machine learning')
      AND b.skill IN ('python','sql','data analysis','excel','aws','machine learning')
    GROUP BY a.skill, b.skill
    ORDER BY co_ocurrencias DESC
    LIMIT 15
""").df()

combos['combo'] = combos['skill_1'] + ' + ' + combos['skill_2']

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=combos, y='combo', x='co_ocurrencias', palette='crest', ax=ax)
ax.set_title('Top 15 Combinaciones de Skills más frecuentes', fontsize=14, fontweight='bold')
ax.set_xlabel('Co-ocurrencias')
ax.set_ylabel('')
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
plt.tight_layout()
plt.show()

### 5.8 Skills por modalidad de trabajo

In [ ]:
skills_modality = con.execute("""
    SELECT js.skill, j.work_type as modalidad, COUNT(*) as total
    FROM job_skills js
    JOIN jobs j ON js.job_link = j.job_link
    WHERE j.work_type IN ('On-site', 'Hybrid', 'Remote')
      AND js.skill IN (SELECT skill FROM job_skills GROUP BY skill ORDER BY COUNT(*) DESC LIMIT 10)
    GROUP BY js.skill, modalidad
""").df()

pivot_mod = skills_modality.pivot(index='skill', columns='modalidad', values='total').fillna(0)

fig, ax = plt.subplots(figsize=(12, 7))
pivot_mod.plot(kind='barh', ax=ax, colormap='tab10')
ax.set_title('Top Skills por Modalidad de Trabajo', fontsize=14, fontweight='bold')
ax.set_xlabel('Número de ofertas')
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
plt.tight_layout()
plt.show()

---
## 6. Conclusiones

### Hallazgos del pipeline ETL

| Métrica | Valor |
|---|---|
| Registros totales del dataset | 1,348,454 |
| Registros filtrados (US) | 1,149,342 |
| Relaciones job→skill generadas | 23,180,623 |
| Duplicados encontrados | 0 |

### Hallazgos del EDA

**Sesgo del dataset:**
- 89% de las ofertas son para nivel Mid-Senior, solo 11% para Associate.
- 99% de las ofertas son modalidad Onsite — el dataset no representa bien el trabajo remoto.

**Skills dominantes:**
- Las soft skills (Communication, Teamwork, Leadership) superan ampliamente a las técnicas en volumen.
- Excel supera a Python en el mercado general, lo que refleja que aún domina en roles no especializados.
- Python, SQL y Data Analysis son las skills técnicas más demandadas para roles de datos.

**Brechas identificadas:**
- Herramientas modernas de ingeniería de datos (dbt, Airflow, Databricks) tienen baja presencia, posiblemente porque son más recientes o se mencionan bajo otros nombres.
- La normalización de skills es un desafío: variantes como 'Python', 'python3', 'Python Programming' deben unificarse para análisis precisos.

### Próximos pasos (Corte 2)

- Clustering de skills para identificar perfiles de trabajo
- Modelos de recomendación de habilidades por rol objetivo
- Predicción de demanda temporal de skills
- Dashboard interactivo con Streamlit
- Servidor MCP para consultas en lenguaje natural

In [ ]:
# Cerrar conexión a DuckDB
con.close()
print('✅ Pipeline completado. Conexión DuckDB cerrada.')